# Preparations

## Imports

In [4]:
import polars as pl

playlists = pl.scan_parquet('../processed_data/data_playlist_metadata.parquet')
playlist_tracks = pl.scan_parquet('../processed_data/data_playlist_songs.parquet')
tracks = pl.scan_parquet('../processed_data/data_song_metadata.parquet')

# Analysis

## Tokenization

In [8]:
def tokenize(expr: pl.Expr) -> pl.Expr:
    return expr.str.to_lowercase().str.split(' ')


def tokenize_unique(expr: pl.Expr) -> pl.Expr:
    return tokenize(expr)\
        .list.filter(pl.element().ne(''))\
        .list.unique(maintain_order=True)


def tokenize_filtered(expr: pl.Expr) -> pl.Expr:
    return (
        tokenize_unique(expr)
        # Filter our years & BPM ranges
        .list.filter(~pl.element().str.contains("^([0-9]+|[0-9]+-[0-9]+)$"))
        # Filter out stuff consisting only of non-letters
        .list.filter(pl.element().str.contains("[[:alpha:]]"))
    )

## Playlist statistics

In [10]:
playlists_tokenized = playlists.select(
    pl.col('playlist.id'),
    pl.col('playlist.name'),
    pl.col('playlist.name').pipe(tokenize_filtered).alias('unique_terms'),
)

exploded_playlists_tokenized = playlists_tokenized\
    .explode('unique_terms')\
    .rename({'unique_terms': 'term'})

tokens = exploded_playlists_tokenized\
    .group_by('term')\
    .agg(pl.col('term').count().alias('playlist_count'),
         pl.col('playlist.name').head(20))\
    .sort('playlist_count', descending=True)

tokens.filter(pl.col('playlist_count').ge(100)).collect(engine='streaming')

term,playlist_count,playlist.name
str,u32,list[str]
"""wcs""",14539,"[""Pop Radio WCS"", ""september wcs"", … ""WCS 2024""]"
"""swing""",2652,"[""MY SWING 候補"", ""West Coast Swing"", … ""Mizzou Swing 9/7""]"
"""blues""",1961,"[""WCS - Blues / R&B / Soul"", ""West Coast Swing Blues"", … ""WCS - Blues (CDF)""]"
"""bpm""",1957,"[""Best WCS by BPM Old"", ""01 WCS - TDLB - BPM 110 à 114"", … ""Oma Alkeet 2022 sorted by increasing BPM""]"
"""songs""",1918,"[""Songs with Swing/Blues Shuffle Timing"", ""Bob Dylan wrote a lot of songs"", … ""Jam songs""]"
…,…,…
"""intermediate""",102,"[""Intermediate WCS"", ""WCS Competition - Intermediate"", … ""J&J intermediate ""]"
"""j&j""",102,"[""Int/Adv J&J Prelims"", ""J&J"", … ""Champs J&J Halloween Swingthing totally stolen from Aiden i think?""]"
"""mar""",102,"[""Torsdagstrening 21 Mar 2024"", ""Mar 2025"", … ""WCS Xchange Mar wk3""]"
